In [25]:
import os
import pandas as pd
import ast
from itertools import chain
from collections import Counter
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm.notebook import tqdm
tqdm.pandas()

In [26]:
db_file_path = '/mnt/share/kaichixie/A2H_preclinical_database/outputs/preclinical_db/v2/a2h_database_version_2.xlsx'
pcdb = pd.read_excel(db_file_path, engine="openpyxl", dtype=str,
                       sheet_name='A2H_database_version_2', keep_default_na=False)

unmapped_extractions = pd.read_excel(db_file_path,
                          sheet_name="unmapped_extractions", engine="openpyxl", dtype=str,
                          keep_default_na=False)

with open("/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_synonyms_look_up_table.pkl", 'rb') as f:
    mesh_synonyms_LUT = pickle.load(f)
    
with open('/mnt/share/kaichixie/A2H_preclinical_database/data/disease_ontology/DO_synonyms_look_up_table.pkl', 'rb') as f:
    DO_synonyms_LUT = pickle.load(f)

pcdb["disease_external_ids"] = pcdb["disease_external_ids"].apply(lambda x: ast.literal_eval(x) if x != '' else x)

In [88]:
pcdb_disease = pcdb.explode("disease_external_ids")
pcdb_disease = pcdb_disease[pcdb_disease['disease_external_ids']!=""]

pcdb_disease["disease_mesh_id"] = \
    pcdb_disease["disease_external_ids"].apply(lambda x : x['mesh'] if ('mesh' in x) else "")
pcdb_disease = pcdb_disease[pcdb_disease['disease_mesh_id']!='']

pcdb_disease = pcdb_disease.explode("disease_mesh_id")
all_mapped_disease_mesh_ids = pcdb_disease['disease_mesh_id'].unique().tolist()

In [89]:
def create_disease_tree_id_lut():
    mesh_desc_data = pd.read_csv("/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_descriptive_data.tsv",
                                 sep="\t")
    mesh_desc_data["tree_numbers"] = \
        mesh_desc_data["tree_numbers"].apply(lambda x: ast.literal_eval(x))
    tree_id_lut = {}
    
    for _, row in mesh_desc_data.iterrows():
        # print(row['mesh_id'])
        tree_id_lut[row['mesh_id']] = row['tree_numbers']
    return tree_id_lut
    
def get_disease_mesh_tree_id(df: pd.DataFrame):
    disease_tree_id_lut = create_disease_tree_id_lut()
    df["disease_mesh_tree_id"] = \
        df["disease_mesh_id"].apply(lambda x: disease_tree_id_lut[x] if (x in disease_tree_id_lut) else "")
    print("Unfound mesh tree id")
    print(len(df[df["disease_mesh_tree_id"] == '']))
    return df

In [90]:
pcdb_disease_entities = pd.DataFrame.from_dict({"disease_mesh_id": all_mapped_disease_mesh_ids})
get_disease_mesh_tree_id(pcdb_disease_entities)

# Only C (disease category) and F03 (Mental disorders) are allowed
pcdb_disease_entities["disease_categories"] = \
    pcdb_disease_entities["disease_mesh_tree_id"].apply(lambda x: [i.split(".")[0] for i in x if (i.startswith('C') or i.startswith('F03'))])

# Clean empty ones
pcdb_disease_entities["disease_categories"] = \
    pcdb_disease_entities["disease_categories"].apply(lambda x: '' if len(x) == 0 else x)
pcdb_disease_entities = pcdb_disease_entities[pcdb_disease_entities["disease_categories"]!=''].copy()
pcdb_disease_entities['disease_categories'] = \
    pcdb_disease_entities['disease_categories'].apply(lambda x: list(set(x)))
# Exclude C22 - animal diseases as we are focusing on human diseases.
pcdb_disease_entities["disease_categories"] = \
    pcdb_disease_entities["disease_categories"].apply(lambda x: "" if ((len(x) == 1) and x[0]=='C22') else x)

Unfound mesh tree id
0


In [91]:
disease_PCDB_ids = [f"PCDB_DI{i}" for i in range(1, len(pcdb_disease_entities)+1)]
pcdb_disease_entities["PCDB_id"] = disease_PCDB_ids

In [92]:
mesh_ids_to_DO_ids = {}
for m_id in pcdb_disease_entities['disease_mesh_id'].unique():
    mesh_ids_to_DO_ids[m_id] = set()

In [93]:
for d_eids in tqdm(pcdb["disease_external_ids"].to_list(), total=len(pcdb)):
    for d_eid in d_eids:
        if 'mesh' in d_eid and 'diseaseontology' in d_eid:
            mesh_ids = d_eid['mesh']
            for m_id in mesh_ids:
                if m_id in mesh_ids_to_DO_ids:
                    for do_id in d_eid['diseaseontology']:
                        mesh_ids_to_DO_ids[m_id].add(do_id)

  0%|          | 0/186627 [00:00<?, ?it/s]

In [94]:
pcdb_disease_entities["diseaseontology_ids"] = \
    pcdb_disease_entities["disease_mesh_id"].apply(lambda x: list(mesh_ids_to_DO_ids[x]) if len(mesh_ids_to_DO_ids[x]) != 0 else "")

In [95]:
DO_ids_linked_mesh = list(chain.from_iterable(pcdb_disease_entities['diseaseontology_ids'].to_list()))
DO_ids_linked_mesh = set(DO_ids_linked_mesh)


In [96]:
pcdb_disease_DO = pcdb.explode("disease_external_ids")
pcdb_disease_DO = pcdb_disease_DO[pcdb_disease_DO['disease_external_ids']!=""]

pcdb_disease_DO["disease_DO_id"] = \
    pcdb_disease_DO["disease_external_ids"].apply(lambda x : x['diseaseontology'] if ('diseaseontology' in x) else "")
pcdb_disease_DO = pcdb_disease_DO[pcdb_disease_DO['disease_DO_id']!='']

pcdb_disease_DO = pcdb_disease_DO.explode("disease_DO_id")
all_mapped_disease_DO_ids = pcdb_disease_DO['disease_DO_id'].unique().tolist()

In [97]:
mapped_disease_DO_ids_not_linked_mesh = [x for x in all_mapped_disease_DO_ids if x not in DO_ids_linked_mesh]

In [98]:
current_id_max = len(pcdb_disease_entities)


In [99]:
for do_id in mapped_disease_DO_ids_not_linked_mesh:
    new_row = pd.DataFrame.from_dict({
        "disease_mesh_id" : "",
        'disease_mesh_tree_id': "",
        'disease_categories': '',
        "PCDB_id": f"PCDB_DI{current_id_max+1}",
        "diseaseontology_ids": list([do_id])
    })
    current_id_max = current_id_max + 1
    pcdb_disease_entities = pd.concat([pcdb_disease_entities, new_row], ignore_index=True)
    

In [100]:
pcdb_disease_entities

,disease_mesh_id,disease_mesh_tree_id,disease_categories,PCDB_id,diseaseontology_ids
0,D015179,"[C04.588.274.476.411.307, C06.301.371.411.307,...","[C04, C06]",PCDB_DI1,"[DOID:9256, DOID:0080199]"
1,D016889,"[C04.588.945.418.948.585, C12.050.351.500.852....","[C04, C12]",PCDB_DI2,"[DOID:1380, DOID:2871]"
2,D006816,"[C10.228.140.079.545, C10.228.140.380.278, C10...","[F03, C16, C10]",PCDB_DI3,[DOID:12858]
3,D001943,"[C04.588.180, C17.800.090.500]","[C04, C17]",PCDB_DI4,"[DOID:1612, DOID:3459]"
4,D018281,[C04.557.470.200.025.450],[C04],PCDB_DI5,"[DOID:4947, DOID:4928]"
...,...,...,...,...,...
2597,,,,PCDB_DI2598,DOID:0080780
2598,,,,PCDB_DI2599,DOID:2174
2599,,,,PCDB_DI2600,DOID:3016
2600,,,,PCDB_DI2601,DOID:849


In [101]:
pcdb_disease_entities["mesh_synonyms"] = pcdb_disease_entities["disease_mesh_id"].apply(lambda x: mesh_synonyms_LUT[x] if x != '' else '')
pcdb_disease_entities["mesh_descriptor_name"] = pcdb_disease_entities["mesh_synonyms"].apply(lambda x: x[0] if x != '' else '')

In [103]:
def retrieve_DO_synonyms(DO_ids):
    if DO_ids == '':
        return ''
    if isinstance(DO_ids, str):
        DO_ids = [DO_ids]
    return [DO_synonyms_LUT[i] for i in DO_ids]

pcdb_disease_entities["diseaseontology_synonyms"] = \
    pcdb_disease_entities['diseaseontology_ids'].apply(retrieve_DO_synonyms)
pcdb_disease_entities["diseaseontology_synonyms"] = \
    pcdb_disease_entities["diseaseontology_synonyms"].apply(lambda x: list(chain.from_iterable(x)) if x!='' else x)

In [105]:
pcdb_disease_entities['disease_name'] = pcdb_disease_entities.apply(lambda x: x['mesh_descriptor_name'] if x['mesh_descriptor_name']!='' else x['diseaseontology_synonyms'][0], 
                                                                    axis=1)

In [106]:
pcdb_disease_entities

,disease_mesh_id,disease_mesh_tree_id,disease_categories,PCDB_id,diseaseontology_ids,mesh_synonyms,mesh_descriptor_name,diseaseontology_synonyms,disease_name
0,D015179,"[C04.588.274.476.411.307, C06.301.371.411.307,...","[C04, C06]",PCDB_DI1,"[DOID:9256, DOID:0080199]","[Colorectal Neoplasms, Colorectal Neoplasm, Ne...",Colorectal Neoplasms,"[colorectal cancer, colorectal carcinoma]",Colorectal Neoplasms
1,D016889,"[C04.588.945.418.948.585, C12.050.351.500.852....","[C04, C12]",PCDB_DI2,"[DOID:1380, DOID:2871]","[Endometrial Neoplasms, Endometrial Neoplasm, ...",Endometrial Neoplasms,"[endometrial neoplasm, neoplasm of endometrium...",Endometrial Neoplasms
2,D006816,"[C10.228.140.079.545, C10.228.140.380.278, C10...","[F03, C16, C10]",PCDB_DI3,[DOID:12858],"[Huntington Disease, Progressive Chorea, Hered...",Huntington Disease,"[HD, Huntington's chorea, Huntington's disease...",Huntington Disease
3,D001943,"[C04.588.180, C17.800.090.500]","[C04, C17]",PCDB_DI4,"[DOID:1612, DOID:3459]","[Breast Neoplasms, Breast Neoplasm, Neoplasm, ...",Breast Neoplasms,"[malignant tumor of the breast, mammary cancer...",Breast Neoplasms
4,D018281,[C04.557.470.200.025.450],[C04],PCDB_DI5,"[DOID:4947, DOID:4928]","[Cholangiocarcinoma, Cholangiocarcinomas, Chol...",Cholangiocarcinoma,"[cholangiosarcoma, cholangiocarcinoma, adult p...",Cholangiocarcinoma
...,...,...,...,...,...,...,...,...,...
2597,,,,PCDB_DI2598,DOID:0080780,,,[acute erythroid leukemia],acute erythroid leukemia
2598,,,,PCDB_DI2599,DOID:2174,,,"[Ocular tumor, ocular cancer, eye neoplasm, ne...",Ocular tumor
2599,,,,PCDB_DI2600,DOID:3016,,,"[breast malignant phyllodes tumor, malignant M...",breast malignant phyllodes tumor
2600,,,,PCDB_DI2601,DOID:849,,,"[Rheumatoid lung, rheumatoid arthritis interst...",Rheumatoid lung


In [108]:
def combine_synonyms(row):
    synonym_groups = [row['mesh_synonyms'], row['diseaseontology_synonyms']]
    all_synonyms = []
    for s_group in synonym_groups:
        if s_group == "":
            continue
        else:
            s_group = [s.lower() for s in s_group]
            all_synonyms = all_synonyms + s_group
    
    if len(all_synonyms) == 0:
        return ''
    
    return list(set(all_synonyms))

pcdb_disease_entities["synonyms"] = pcdb_disease_entities.apply(combine_synonyms, axis=1)

In [109]:
pcdb_disease_entities

,disease_mesh_id,disease_mesh_tree_id,disease_categories,PCDB_id,diseaseontology_ids,mesh_synonyms,mesh_descriptor_name,diseaseontology_synonyms,disease_name,synonyms
0,D015179,"[C04.588.274.476.411.307, C06.301.371.411.307,...","[C04, C06]",PCDB_DI1,"[DOID:9256, DOID:0080199]","[Colorectal Neoplasms, Colorectal Neoplasm, Ne...",Colorectal Neoplasms,"[colorectal cancer, colorectal carcinoma]",Colorectal Neoplasms,"[colorectal neoplasms, colorectal carcinoma, c..."
1,D016889,"[C04.588.945.418.948.585, C12.050.351.500.852....","[C04, C12]",PCDB_DI2,"[DOID:1380, DOID:2871]","[Endometrial Neoplasms, Endometrial Neoplasm, ...",Endometrial Neoplasms,"[endometrial neoplasm, neoplasm of endometrium...",Endometrial Neoplasms,"[endometrial carcinoma, endometrium carcinoma,..."
2,D006816,"[C10.228.140.079.545, C10.228.140.380.278, C10...","[F03, C16, C10]",PCDB_DI3,[DOID:12858],"[Huntington Disease, Progressive Chorea, Hered...",Huntington Disease,"[HD, Huntington's chorea, Huntington's disease...",Huntington Disease,"[juvenile-onset huntington disease, huntington..."
3,D001943,"[C04.588.180, C17.800.090.500]","[C04, C17]",PCDB_DI4,"[DOID:1612, DOID:3459]","[Breast Neoplasms, Breast Neoplasm, Neoplasm, ...",Breast Neoplasms,"[malignant tumor of the breast, mammary cancer...",Breast Neoplasms,"[tumor, breast, human mammary neoplasm, mammar..."
4,D018281,[C04.557.470.200.025.450],[C04],PCDB_DI5,"[DOID:4947, DOID:4928]","[Cholangiocarcinoma, Cholangiocarcinomas, Chol...",Cholangiocarcinoma,"[cholangiosarcoma, cholangiocarcinoma, adult p...",Cholangiocarcinoma,"[cholangiocarcinoma, extrahepatic, cholangioce..."
...,...,...,...,...,...,...,...,...,...,...
2597,,,,PCDB_DI2598,DOID:0080780,,,[acute erythroid leukemia],acute erythroid leukemia,[acute erythroid leukemia]
2598,,,,PCDB_DI2599,DOID:2174,,,"[Ocular tumor, ocular cancer, eye neoplasm, ne...",Ocular tumor,"[neoplasm of eye, ocular tumor, malignant eye ..."
2599,,,,PCDB_DI2600,DOID:3016,,,"[breast malignant phyllodes tumor, malignant M...",breast malignant phyllodes tumor,"[malignant phyllodes tumor, phyllodes breast t..."
2600,,,,PCDB_DI2601,DOID:849,,,"[Rheumatoid lung, rheumatoid arthritis interst...",Rheumatoid lung,[rheumatoid arthritis interstitial lung diseas...


In [110]:
pcdb_disease_entities.columns

Index(['disease_mesh_id', 'disease_mesh_tree_id', 'disease_categories',
       'PCDB_id', 'diseaseontology_ids', 'mesh_synonyms',
       'mesh_descriptor_name', 'diseaseontology_synonyms', 'disease_name',
       'synonyms'],
      dtype='object')

In [111]:
pcdb_disease_entities = pcdb_disease_entities[['PCDB_id', 'disease_name', 'synonyms', 
                                               'mesh_descriptor_name', 
                                               'disease_mesh_id','diseaseontology_ids',
                                               'mesh_synonyms','diseaseontology_synonyms',
                                               'disease_mesh_tree_id', 'disease_categories']]

In [112]:
pcdb_disease_entities.to_csv(os.path.join('/mnt/share/kaichixie/A2H_preclinical_database/outputs/preclinical_db/v3',
                                          "disease_entities.tsv"),
                                          sep='\t',
                                          index=False)

In [ ]:
# disease_pcdb_id_LUT = {}
# for i, row in pcdb_disease_entities.iterrows():
#     disease_pcdb_id_LUT[row['disease_mesh_id']] = row['PCDB_id']

# def find_disease_pcdb_ids(d_eids):
#     if d_eids == '':
#         return ''
#     a2h_ids = []
#     for d_eid in d_eids:
#         if 'mesh' in d_eid:
#             for m_id in d_eid['mesh']:
#                 if m_id in disease_pcdb_id_LUT:
#                     a2h_ids.append(disease_pcdb_id_LUT[m_id])
#     if len(a2h_ids) == "":
#         return ""
#     return list(set(a2h_ids))
    
# pcdb["disease_PCDB_ids"] = \
#     pcdb["disease_external_ids"].apply(find_disease_pcdb_ids)
# pcdb_disease_only = pcdb[['pmcid', 'date', 'link', 'confidence_score', 'TITLE', 'disease_PCDB_ids']]
# pcdb_disease_only = pcdb_disease_only.drop_duplicates(['pmcid'])
# pcdb_disease_only = pcdb_disease_only[pcdb_disease_only['disease_PCDB_ids'].apply(lambda x: len(x)>0)]